   # Trabajo Práctico 1 (2026 - 2do semestre)

In [11]:
import pandas as pd

df = pd.read_csv("datatelco_customer_churn.csv")

pd.to_numeric(df['TotalCharges'], errors='coerce')
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [12]:
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

# El problema con las columnas que no son reconocidas directamente como numéricas

Luego de investigar un poco, descubrimos que Pandas infiere el dtype al leer el CSV, columna por columna, y aplica una regla simple: si todos los valores de la columna se pueden convertir a número, la hace numérica pero si aparece aunque sea uno solo que no puede, toda la columna cae a object.Object significa que la columna no guarda los valores en sí, sino punteros a objetos de Python. En la práctica, casi siempre son strings.

El caso que encontramos fue con TotalCharges. Hay 7043 filas, de las cuales 7032 son montos perfectamente numéricos. Pero 11 tienen un espacio en blanco. Ese espacio no se puede convertir a número, así que pandas se rinde y guarda las 7043 como strings. Un "29.85" con comillas, no un 29.85.

Los espacios en blanco aparecen cuando son clientes con tenure = 0, o sea que recién se dieron de alta y todavía no facturaron nada. Esa columna sí es numérica conceptualmente, pero pandas la lee como object porque hay unas 11 filas con un string vacío (" ") en lugar de un número.

El impacto de no detectarlo es que la columna queda como texto y el modelo o la ignora, o si alguien la codifica como categórica, termina tratando cada monto como una categoría distinta.

Se arregla con pd.to_numeric(df['TotalCharges'], errors='coerce') y después tendríamos que decidir qué hacer con esos NaN (imputar con 0 tiene sentido acá, justificando que no facturaron todavía).

In [13]:
# 1. Conversión: los " " no parseables pasan a NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 2. Verificación del diagnóstico (para dejar registro en el notebook)
faltantes = df['TotalCharges'].isna()
print(f"Filas con TotalCharges nulo: {faltantes.sum()}")
print(f"Valores de tenure en esas filas: {df.loc[faltantes, 'tenure'].unique()}")

# 3. Chequeo final
print(f"\nDtype: {df['TotalCharges'].dtype}")
print(f"Nulos restantes: {df['TotalCharges'].isna().sum()}")

df.dtypes

Filas con TotalCharges nulo: 11
Valores de tenure en esas filas: [0]

Dtype: float64
Nulos restantes: 11


customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

## 2. Preprocesamiento y prevención de Data Leakage

### **Valores faltantes**

Compruebo que el único problema de datos faltantes en todo el dataset fue el de TotalCharges, lo cual se deja constancia a continuación.

In [14]:
# Nulos "reales" en todas las columnas
print(df.isnull().sum()[df.isnull().sum() > 0])

# Nulos compuestos por "texto raro" en columnas categóricas
for col in df.select_dtypes(include='object').columns:
    valores_raros = df[col].apply(lambda x: isinstance(x, str) and x.strip() == '').sum()
    if valores_raros > 0:
        print(f"{col}: {valores_raros} valores vacíos/espacios")

TotalCharges    11
dtype: int64


Para las filas con `TotalCharges` nulo y `tenure=0`, imputo con 0, ya que son clientes que recién se dieron de alta y todavía no generaron facturación acumulada.

Si quedaran filas con `TotalCharges` nulo y `tenure>0` (verificado arriba), esos casos se imputan luego (para evitar data leakage)

In [15]:
# Imputación condicional (justificada por lógica de negocio)
condicion_tenure_cero = (df['TotalCharges'].isna()) & (df['tenure'] == 0)
condicion_otros_nulos = (df['TotalCharges'].isna()) & (df['tenure'] != 0)

print(f"Nulos con tenure = 0: {condicion_tenure_cero.sum()}")
print(f"Nulos con tenure > 0: {condicion_otros_nulos.sum()}")

df.loc[condicion_tenure_cero, 'TotalCharges'] = 0

print(f"Nulos restantes en TotalCharges: {df['TotalCharges'].isna().sum()}")

Nulos con tenure = 0: 11
Nulos con tenure > 0: 0
Nulos restantes en TotalCharges: 0


In [16]:
# Imputación para el caso tenure=0: TotalCharges = 0 (lógica de negocio: el cliente todavía no facturó nada)
df.loc[condicion_tenure_cero, 'TotalCharges'] = 0

# Chequeo: cuántos nulos quedan sin resolver (los de tenure>0, si los hubiera)
print(f"Nulos restantes en TotalCharges: {df['TotalCharges'].isna().sum()}")

Nulos restantes en TotalCharges: 0


### **Variables categóricas**

Al revisar las categorías de las variables de texto, encontramos que varias columnas no son binarias en apariencia, pero sí lo son en esencia:

- `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`,`StreamingMovies` tienen 3 valores: `Yes`, `No`, `No internet service`.
- `MultipleLines` tiene 3 valores: `Yes`, `No`, `No phone service`.

El problema es que la tercera categoría no aporta información nueva: un cliente tiene `"No internet service"` en esas 6 columnas si y solo si `InternetService == "No"`, y tiene `"No phone service"` en `MultipleLines` si y solo si `PhoneService == "No"`. Es decir, esa categoría es perfectamente redundante con una variable que ya está en el dataset.

Por eso, decidimos colapsar `"No internet service"` y `"No phone service"` en `"No"`. Esto convierte a las 7 columnas mencionadas en verdaderamente binarias, permitiendo aplicar Label Encoding de forma consistente con el resto de las variables Yes/No, y sin perder información (porque esa información ya vive en `InternetService` y `PhoneService`).

In [17]:
# Colapsamos "No internet service" / "No phone service" en "No"
cols_no_internet = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in cols_no_internet:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Verificación
for col in cols_no_internet + ['MultipleLines']:
    print(f"{col}: {df[col].unique()}")

OnlineSecurity: ['No' 'Yes']
OnlineBackup: ['Yes' 'No']
DeviceProtection: ['No' 'Yes']
TechSupport: ['No' 'Yes']
StreamingTV: ['No' 'Yes']
StreamingMovies: ['No' 'Yes']
MultipleLines: ['No' 'Yes']


In [18]:
#Ahora aplico One Hot Encoding y Label Encoding a las columnas categóricas
from sklearn.preprocessing import LabelEncoder

# customerID es un identificador único por cliente, no una variable predictiva. Si no hacemos este paso, 
# el One-Hot Encoding lo trataría como una columna categórica
df = df.drop(columns=['customerID'])

# 1. Separamos las columnas categóricas en binarias (2 categorías) y multiclase (3+),
# dejando Churn afuera porque es el target y lo tratamos aparte
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('Churn')

binarias = [col for col in cat_cols if df[col].nunique() == 2]
multiclase = [col for col in cat_cols if df[col].nunique() > 2]

#Verificación
print("Binarias (Label Encoding):", binarias)
print("Multiclase (One-Hot Encoding):", multiclase)

Binarias (Label Encoding): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']
Multiclase (One-Hot Encoding): ['InternetService', 'Contract', 'PaymentMethod']


In [19]:
# 2. Label Encoding para las binarias
le = LabelEncoder()
mapeos = {}  # guardamos el mapeo de cada columna para dejar constancia

for col in binarias:
    df[col] = le.fit_transform(df[col])
    mapeos[col] = dict(zip(le.classes_, le.transform(le.classes_)))

for col, mapeo in mapeos.items():
    print(f"{col}: {mapeo}")

# 3. One-Hot Encoding para las que tienen 3 o más categorías 
df = pd.get_dummies(df, columns=multiclase, drop_first=True)

# 4. Encodeamos el target por separado (No -> 0, Yes -> 1)
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

# 5. Chequeo final
print(df.dtypes.value_counts())
print(df.select_dtypes(include='object').columns)

# Convertimos columnas booleanas a enteros para mejor compatibilidad con modelos de ML
dummy_cols = df.select_dtypes(include='bool').columns
df[dummy_cols] = df[dummy_cols].astype(int)

gender: {'Female': np.int64(0), 'Male': np.int64(1)}
Partner: {'No': np.int64(0), 'Yes': np.int64(1)}
Dependents: {'No': np.int64(0), 'Yes': np.int64(1)}
PhoneService: {'No': np.int64(0), 'Yes': np.int64(1)}
MultipleLines: {'No': np.int64(0), 'Yes': np.int64(1)}
OnlineSecurity: {'No': np.int64(0), 'Yes': np.int64(1)}
OnlineBackup: {'No': np.int64(0), 'Yes': np.int64(1)}
DeviceProtection: {'No': np.int64(0), 'Yes': np.int64(1)}
TechSupport: {'No': np.int64(0), 'Yes': np.int64(1)}
StreamingTV: {'No': np.int64(0), 'Yes': np.int64(1)}
StreamingMovies: {'No': np.int64(0), 'Yes': np.int64(1)}
PaperlessBilling: {'No': np.int64(0), 'Yes': np.int64(1)}
int64      15
bool        7
float64     2
Name: count, dtype: int64
Index([], dtype='object')


### Feature scaling
Utilizamos StandardScaler (completar por qué)

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Separamos features y target
X = df.drop(columns=['Churn'])
y = df['Churn']

# 2. Train/Test split ANTES de escalar, para evitar data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Columnas numéricas continuas a escalar (las binarias/dummies ya son 0/1, no hace falta)
cols_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']

# 4. Fiteamos el scaler SOLO con X_train
scaler = StandardScaler()
X_train[cols_numericas] = scaler.fit_transform(X_train[cols_numericas])
X_test[cols_numericas] = scaler.transform(X_test[cols_numericas])  # solo transform, sin fit

X_train[cols_numericas].describe()

,tenure,MonthlyCharges,TotalCharges
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,-1.008935e-17,-2.402527e-16,2.522338e-17
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.322329e+00,-1.544028e+00,-1.008922e+00
25%,-9.559779e-01,-9.711977e-01,-8.321009e-01
50%,-1.418632e-01,1.848336e-01,-3.968446e-01
75%,9.164859e-01,8.319124e-01,6.741944e-01
max,1.608483e+00,1.785939e+00,2.801869e+00


## Datos de entrenamiento SIN escalar

Ojo con esto: en tu notebook, X_train ya quedó escalado a mano en el punto 2. Para el punto 3 necesitamos que el pipeline sea el que escale, así que le tenemos que dar los datos sin escalar. Como X (antes del split) nunca se escaló, simplemente rehacemos el split:

In [21]:
from sklearn.model_selection import train_test_split

# Rehacemos el split partiendo de X sin escalar.
# Con el MISMO random_state=42, las filas que caen en train y test son
# exactamente las mismas de antes (misma partición, reproducible).
X = df.drop(columns=['Churn'])   # todas las features (ya numéricas), SIN escalar
y = df['Churn']                  # el target: 0 = no se va, 1 = se va

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y # estratifica. Mantiene la misma proporción de "se va / no se va" en train y en test. 
)                                                    # Como tu clase está desbalanceada, esto importa.

## Armamos el preprocesador (el escalado por columnas)

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Las columnas numéricas continuas que SÍ hay que escalar
cols_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']

# ColumnTransformer: aplica StandardScaler solo a esas 3 columnas.
# El resto (binarias y dummies) pasa sin cambios gracias a remainder='passthrough'.
preprocesador = ColumnTransformer(
    transformers=[('escalado', StandardScaler(), cols_numericas)],
    remainder='passthrough'
)

## Acá armamos el Pipeline completo (preprocesado + modelo)

* steps=[...]: la secuencia de pasos, en orden. Cada paso es una tupla (nombre, objeto).
    - Primero 'preprocesamiento' → el ColumnTransformer de la celda B (escala las numéricas).
    - Después 'modelo' → la regresión logística.
* LogisticRegression(...): crea el clasificador.
    - max_iter=1000: la regresión logística encuentra sus pesos mediante un proceso iterativo (va mejorando de a pasos hasta "converger", o sea hasta que deja de mejorar). El valor por defecto (100 pasos) a veces no alcanza y tira un warning de "no convergió". Subirlo a 1000 le da margen para terminar tranquilo. No cambia el concepto, solo evita el warning.
    - random_state=42: semilla, para reproducibilidad del propio modelo.

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', LogisticRegression(max_iter=1000, random_state=42))
])

## Definimos el esquema de K-Fold estratificado

* StratifiedKFold: la versión de K-Fold que mantiene la proporción de clases en cada fold. Como tu Churn está desbalanceado, es la correcta (y la consigna te pide justificar justo esto).
* n_splits=5: los 5 folds que pide el TP (la "K" de K-Fold).
* shuffle=True: baraja las filas antes de cortar. Sin esto, cortaría los folds en el orden en que vienen las filas, y si el dataset tuviera algún orden oculto, los folds
saldrían sesgados. Barajar lo evita.
* random_state=42: semilla, para que el barajado sea siempre igual (reproducible).

In [24]:
from sklearn.model_selection import StratifiedKFold

# 5 folds, estratificado (respeta la proporción de clases en cada fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Ejecutamos la validación cruzada

Acá es donde por fin corre todo. Usamos una función que hace el loop de las 5 rondas por nosotros:

In [25]:
from sklearn.model_selection import cross_validate

# Las métricas que queremos medir en cada fold
metricas = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

resultados = cross_validate(
    pipeline,      # la cadena escalado+modelo que armamos
    X_train,       # features de entrenamiento (SIN escalar: el pipeline escala)
    y_train,       # target de entrenamiento
    cv=skf,        # el esquema de 5 folds estratificados
    scoring=metricas
)

## Resumimos: promedio y desvío por métrica

* valores.mean(): el promedio de los 5 folds → tu estimación de qué tan bien generaliza.
* valores.std(): el desvío estándar → qué tan parejo fue entre folds. Chico = estable; grande = sensible a la partición.
* El :.4f es solo formato (4 decimales).

In [26]:
for m in metricas:
    valores = resultados['test_' + m]   # los 5 valores de esta métrica
    print(f"{m:12s}: {valores.mean():.4f}  (+/- {valores.std():.4f})")

accuracy    : 0.8025  (+/- 0.0124)
precision   : 0.6541  (+/- 0.0281)
recall      : 0.5431  (+/- 0.0411)
f1          : 0.5928  (+/- 0.0304)
roc_auc     : 0.8460  (+/- 0.0126)


Por qué K-Fold en vez de un simple Holdout: "Holdout" es apartar un solo pedazo para validar, una sola vez. El problema es que tu estimación depende de qué filas cayeron en ese pedazo (suerte). K-Fold usa todas las filas para validar (cada una una vez, a lo largo de las 5 rondas) y promedia, dando una estimación más robusta y menos dependiente del azar. Además, al reportar el desvío, ves cuán estable es el modelo.

Por qué Stratified K-Fold: tu Churn está desbalanceado (bastante más "No" que "Yes"). Con un K-Fold común, por azar un fold podría quedar con muy pocos "Yes", y su validación sería poco representativa (y métricas como Precision/Recall se volverían inestables). Estratificar fuerza a que cada fold tenga la misma proporción de clases que el total, así cada ronda evalúa sobre una muestra representativa.

# 4. Evaluación del modelo base